# Case study · Five questions about the reviews

*Session 1, block 2 practice. Author: course team, licence CC-BY-4.0.*

You answer five questions about the stratified sample of 50,000 Amazon health product reviews with pandas. Write each answer as **one sentence** in the Markdown cell below the code that produces it.

**Before you start**, generate the data once from the repository root:

```bash
uv run --with pandas --with pyarrow python case-study/prepare_data.py
```

Then start JupyterLab from the repository root, for example with
`uv run --with pandas --with pyarrow --with jupyterlab jupyter lab`, and open this notebook.

The questions:

1. How many reviews and columns are there, and which period do the dates cover?
2. What share of reviews is positive, neutral and negative?
3. What share of reviews comes from verified purchases, and does it differ between negative and positive reviews?
4. What is the median length of the review text in characters for each label?
5. Which product (`parent_asin`) has the most reviews in the sample, and what is its mean rating?

Questions 1 and 2 are worked examples. Questions 3 to 5 have hints; a reference solution is at the end. Try first.

## Setup: find the data and load it

In [ ]:
from pathlib import Path

import pandas as pd

# The notebook may be started from the repository root or from its own folder.
# Walk up the folders until we find case-study/data/.
here = Path.cwd().resolve()
root = next(p for p in [here, *here.parents] if (p / "case-study" / "data").exists())
DATA = root / "case-study" / "data" / "train_sample.parquet"

reviews = pd.read_parquet(DATA)
reviews.head(3)

In [ ]:
reviews.dtypes   # check the types first: rating is an integer, date a timestamp, label text

## Question 1 · Size and period (worked example)

In [ ]:
n_rows, n_cols = reviews.shape
first, last = reviews["date"].min(), reviews["date"].max()
print(f"{n_rows:,} reviews, {n_cols} columns, from {first:%Y-%m-%d} to {last:%Y-%m-%d}")

*Answer:* The sample has 50,000 reviews in 11 columns, written between September 2001 and December 2021.

## Question 2 · Label shares (worked example)

In [ ]:
shares = reviews["label"].value_counts(normalize=True).round(3)
shares

*Answer:* About 73 % of the reviews are positive, 19 % negative and 7.5 % neutral.

> **Why it matters.** A model that always predicts "positive" is right 73 % of the time. This is why the course leaderboard uses macro-F1, which gives the three classes equal weight (Session 8).

## Question 3 · Verified purchases

*Hints:* `verified_purchase` is a boolean column; the mean of a boolean column is the share of `True`. For the comparison, group by `label`: `reviews.groupby("label")["verified_purchase"].mean()`.

In [ ]:
# your code for question 3


*Answer:* ...

## Question 4 · Text length by label

*Hints:* `reviews["text"].str.len()` gives the number of characters of each text. Group this Series by `reviews["label"]` and take the `median()`. Why the median and not the mean? Look at the maximum length.

In [ ]:
# your code for question 4


*Answer:* ...

## Question 5 · The most-reviewed product

*Hints:* `value_counts()` on `parent_asin` sorts products by their number of reviews. Select the rows of the first product with a boolean mask and take the mean of `rating`. Optional: look up its title in `products.parquet` with the same `parent_asin`.

In [ ]:
# your code for question 5


*Answer:* ...

## Checking AI-generated code

If you used an AI coding assistant, check its code before you trust the number:

- Compute one value in a second, independent way (for example a filter and `len()` instead of `value_counts()`).
- Try an edge case: what does the code do with an empty DataFrame or a missing value?
- Record the tool, the prompt and how you checked the answer in your notes (later in `AI_USE.md` of your team repository).

In [ ]:
# Example of an independent check for question 2: count the negative reviews with a mask
is_neg = reviews["rating"] <= 2
assert is_neg.sum() == (reviews["label"] == "neg").sum()
is_neg.mean().round(3)

---

## Reference solution

Try the questions yourself first. The cells below compute all five answers.

In [ ]:
# Question 3
print("all reviews:", reviews["verified_purchase"].mean().round(3))
reviews.groupby("label")["verified_purchase"].mean().round(3)

In [ ]:
# Question 4
lengths = reviews["text"].str.len()
print("longest text:", lengths.max(), "characters")   # a few very long texts pull the mean up
lengths.groupby(reviews["label"]).median()

In [ ]:
# Question 5
counts = reviews["parent_asin"].value_counts()
top = counts.index[0]
mean_rating = reviews.loc[reviews["parent_asin"] == top, "rating"].mean()
print(top, counts.iloc[0], "reviews, mean rating", round(mean_rating, 2))

products = pd.read_parquet(root / "case-study" / "data" / "products.parquet")
products.loc[products["parent_asin"] == top, ["title", "store"]]

## Next step

In block 3 the same analysis becomes a module with a class, `ReviewTable`, in [`../workspace/`](../workspace/). Its methods answer these five questions, and tests check them on six hand-made reviews.